In [59]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [124]:
TIME_PERIODS = {
    "full_time":["1992-01-03", "2000-04-20"],
    "sub1":["1992-01-03", "1997-05-30"],
    "sub2":["1997-06-02", "2000-04-20"]
    }


In [60]:
import pandas as pd
import numpy as np

In [61]:
def process_string_csv_data(csv_path):
    out_df = pd.read_csv(
        csv_path,
        header=0,
        names=["Date", "Price", "Open", "High", "Low", "Vol", "Change_%"],
    )

    out_df["Date"] = pd.to_datetime(out_df["Date"], format="%d/%m/%Y")

    for col in out_df.columns[1:]:
        out_df[col] = (
            out_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)  
            .str.replace("%", "", regex=False)  
            .str.replace("M", "", regex=False)  
            .str.strip()
        )
        out_df[col] = pd.to_numeric(out_df[col], errors="coerce")
    return out_df


In [62]:
dji = process_string_csv_data("dji.csv")
print(dji.info())
print('\n-------------------\n')
print(dji.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2681 entries, 0 to 2680
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      2681 non-null   datetime64[us]
 1   Price     2681 non-null   float64       
 2   Open      2681 non-null   float64       
 3   High      2681 non-null   float64       
 4   Low       2681 non-null   float64       
 5   Vol       2101 non-null   float64       
 6   Change_%  2681 non-null   float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 146.7 KB
None

-------------------

                             Date         Price          Open          High  \
count                        2681   2681.000000   2681.000000   2681.000000   
mean   1996-06-19 08:25:57.627750   6420.434946   6417.801063   6449.283021   
min           1992-01-01 00:00:00   3136.580000   3149.010000   3168.830000   
25%           1994-07-22 00:00:00   3801.460000   3805.940000   3813.170000   
50%     

In [63]:
dgs = pd.read_csv('DGS10.csv')
dgs["observation_date"] = pd.to_datetime(dgs["observation_date"], format="%Y-%m-%d")
dgs = dgs.rename(columns={"observation_date":"Date"})
print(dgs.info())
print(dgs.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2169 entries, 0 to 2168
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2169 non-null   datetime64[us]
 1   DGS10   2082 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 34.0 KB
None
                      Date        DGS10
count                 2169  2082.000000
mean   1996-02-28 00:00:00     6.283079
min    1992-01-02 00:00:00     4.160000
25%    1994-01-31 00:00:00     5.770000
50%    1996-02-28 00:00:00     6.250000
75%    1998-03-27 00:00:00     6.790000
max    2000-04-25 00:00:00     8.050000
std                    NaN     0.753005


In [64]:
dji_prices = dji[["Date", "Price"]]
df_us = pd.merge(dji_prices, dgs, on="Date", how="inner")
df_us.head()

,Date,Price,DGS10
0,2000-04-25,11124.83,6.14
1,2000-04-24,10906.10,6.00
2,2000-04-21,10844.06,NaN
3,2000-04-20,10844.06,5.99
4,2000-04-19,10674.97,5.99


In [65]:
def check_nans(df):
    print("Количество пропусков:")
    print(df.isna().sum())

    print("\nПроцент пропусков:")
    print((df.isna().mean() * 100).round(2))

    total_nans = df.isna().sum().sum()
    print(f"\nВсего NaN в таблице: {total_nans}")

check_nans(df_us)
df_us_interp = df_us.ffill()
check_nans(df_us_interp)

Количество пропусков:
Date      0
Price     0
DGS10    87
dtype: int64

Процент пропусков:
Date     0.00
Price    0.00
DGS10    4.01
dtype: float64

Всего NaN в таблице: 87
Количество пропусков:
Date     0
Price    0
DGS10    0
dtype: int64

Процент пропусков:
Date     0.0
Price    0.0
DGS10    0.0
dtype: float64

Всего NaN в таблице: 0


In [66]:
df_us_interp = df_us_interp.rename(columns={"DGS10":"BondYield"})

In [ ]:
def process_colums(df, country_prefix: str, include_duration_method=True):
    df = df.sort_values("Date").reset_index(drop=True).copy()
    df[f"s_{country_prefix}"] = 100 * (np.log(df["Price"]) - np.log(df["Price"].shift(1)))
    df[f"i_{country_prefix}"] = df["BondYield"] / 365.0
    df[f"delta_i_{country_prefix}"] = df["BondYield"] - df["BondYield"].shift(1)
    if include_duration_method:
        df[f"Y_{country_prefix}"] = -7.5 * df[f"delta_i_{country_prefix}"]
    else:
        pass
    df[f"ER_{country_prefix}"] = df[f"s_{country_prefix}"] - df[f"i_{country_prefix}"]
    return df

In [68]:
df_us_processed = process_colums(df_us_interp, "US")
df_us_processed.head(10)

1


,Date,Price,BondYield,s_US,i_US,delta_i_US,Y_US,ER_US
0,1992-01-02,3172.41,6.78,NaN,0.018575,NaN,NaN,NaN
1,1992-01-03,3201.47,6.85,0.911853,0.018767,0.07,-0.525,0.893086
2,1992-01-06,3200.13,6.82,-0.041865,0.018685,-0.03,0.225,-0.060549
3,1992-01-07,3204.83,6.76,0.146761,0.018521,-0.06,0.450,0.128241
4,1992-01-08,3203.93,6.77,-0.028087,0.018548,0.01,-0.075,-0.046635
5,1992-01-09,3209.53,6.79,0.174633,0.018603,0.02,-0.150,0.156030
6,1992-01-10,3199.46,6.85,-0.314246,0.018767,0.06,-0.450,-0.333013
7,1992-01-13,3185.60,6.92,-0.434139,0.018959,0.07,-0.525,-0.453098
8,1992-01-14,3246.20,7.03,1.884443,0.019260,0.11,-0.825,1.865182
9,1992-01-15,3258.50,7.05,0.378189,0.019315,0.02,-0.150,0.358873


In [72]:
df_us_processed = df_us_processed.dropna()
check_nans(df_us_processed)
df_us_processed.describe()

Количество пропусков:
Date          0
Price         0
BondYield     0
s_US          0
i_US          0
delta_i_US    0
Y_US          0
ER_US         0
dtype: int64

Процент пропусков:
Date          0.0
Price         0.0
BondYield     0.0
s_US          0.0
i_US          0.0
delta_i_US    0.0
Y_US          0.0
ER_US         0.0
dtype: float64

Всего NaN в таблице: 0


,Date,Price,BondYield,s_US,i_US,delta_i_US,Y_US,ER_US
count,2168,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000,2168.000000
mean,1996-02-28 16:48:15.940959,6110.562394,6.282265,0.057873,0.017212,-0.000295,0.002214,0.040661
min,1992-01-03 00:00:00,3136.580000,4.160000,-7.454915,0.011397,-0.230000,-2.925000,-7.471106
25%,1994-01-31 18:00:00,3699.072500,5.770000,-0.373440,0.015808,-0.030000,-0.225000,-0.390833
50%,1996-02-28 12:00:00,5474.600000,6.240000,0.031835,0.017096,0.000000,-0.000000,0.013994
75%,1998-03-27 18:00:00,8255.992500,6.790000,0.535662,0.018603,0.030000,0.225000,0.520018
max,2000-04-25 00:00:00,11722.980000,8.050000,4.860535,0.022055,0.390000,1.725000,4.846727
std,NaN,2646.107577,0.755120,0.899431,0.002069,0.057842,0.433812,0.899493


In [128]:
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

def calculate_table1_stats(series):
    s = series.dropna()
    
    stats = {
        "Mean": s.mean(),
        "Median": s.median(),
        "Std. Dev.": s.std(),
        "Skewness": skew(s),
        "Kurtosis": kurtosis(s, fisher=False) 
    }

    jb_stat, jb_pvalue = jarque_bera(s)
    stats["JB"] = jb_stat

    lb_test = acorr_ljungbox(s, lags=[1, 5], return_df=True)
    stats["LB(1)"] = lb_test['lb_stat'].iloc[0]
    stats["LB(5)"] = lb_test['lb_stat'].iloc[1]

    adf_result = adfuller(s, maxlag=10, autolag=None, result_object=False)
    stats["ADF(10)"] = adf_result[0]

    return pd.Series(stats)


In [129]:
df_us_processed_full_time = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["full_time"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["full_time"][1])].copy()
df_us_processed_sub1 = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["sub1"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["sub1"][1])].copy()
df_us_processed_sub2 = df_us_processed[(df_us_processed["Date"] >= TIME_PERIODS["sub2"][0]) & (df_us_processed["Date"] <= TIME_PERIODS["sub2"][1])].copy()
table_1_panel_a_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(df_us_processed_full_time["ER_US"]),
    "YUS": calculate_table1_stats(df_us_processed_full_time["Y_US"]),
    "DLDJIA": calculate_table1_stats(df_us_processed_full_time["s_US"]),
})
table_1_panel_b_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(df_us_processed_sub1["ER_US"]),
    "YUS": calculate_table1_stats(df_us_processed_sub1["Y_US"]),
    "DLDJIA": calculate_table1_stats(df_us_processed_sub1["s_US"]),
})
table_1_panel_c_us = pd.DataFrame({
    "ERUS": calculate_table1_stats(df_us_processed_sub2["ER_US"]),
    "YUS": calculate_table1_stats(df_us_processed_sub2["Y_US"]),
    "DLDJIA": calculate_table1_stats(df_us_processed_sub2["s_US"]),
})
print(f"panel a:{TIME_PERIODS["full_time"]}\n")
print(table_1_panel_a_us.round(4))
print("--------------")
print(f"panel b:{TIME_PERIODS["sub1"]}\n")
print(table_1_panel_b_us.round(4))
print("--------------")
print(f"panel c:{TIME_PERIODS["sub2"]}\n")
print(table_1_panel_c_us.round(4))

panel a:['1992-01-03', '2000-04-20']

                ERUS       YUS     DLDJIA
Mean          0.0396    0.0027     0.0568
Median        0.0136    0.0000     0.0312
Std. Dev.     0.8991    0.4335     0.8990
Skewness     -0.5422   -0.4105    -0.5448
Kurtosis      9.5364    5.7033     9.5418
JB         3960.1681  720.0432  3967.5808
LB(1)         0.7588   13.3322     0.7501
LB(5)        10.3578   30.3081    10.3875
ADF(10)     -14.4601  -13.8435   -14.4707
--------------
panel b:['1992-01-03', '1997-05-30']

               ERUS       YUS    DLDJIA
Mean         0.0413    0.0006    0.0594
Median       0.0213    0.0000    0.0390
Std. Dev.    0.6688    0.4419    0.6687
Skewness    -0.2809   -0.5301   -0.2804
Kurtosis     4.6908    6.3696    4.6919
JB         186.6380  733.5861  186.7801
LB(1)        1.9111    9.2371    1.8936
LB(5)       10.3672   21.4664   10.3947
ADF(10)    -11.8017  -11.3678  -11.8124
--------------
panel c:['1997-06-02', '2000-04-20']

               ERUS      YUS    DLDJ

Дальше другие страны

In [106]:
ftse = pd.read_csv("FTSE.csv")
ftse = ftse.rename(columns={" Close":"Price"})
ftse_prices = ftse[["Date", "Price"]]
ftse_prices.head()

,Date,Price
0,04/25/00,6282.97
1,04/20/00,6241.22
2,04/19/00,6184.91
3,04/18/00,6074.04
4,04/17/00,5994.57


In [107]:
ftse_prices["Date"] = pd.to_datetime(ftse_prices["Date"], format="%m/%d/%y")
ftse_prices.info()


<class 'pandas.DataFrame'>
RangeIndex: 2101 entries, 0 to 2100
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    2101 non-null   datetime64[us]
 1   Price   2101 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 33.0 KB


In [108]:
uk_10y = pd.read_csv("uk_10y_full.csv")
uk_10y = uk_10y.rename(columns={"date":"Date", "yield_10.0y":"BondYield"})
uk_10y["Date"] = pd.to_datetime(uk_10y["Date"], format="%Y-%m-%d")
print(uk_10y.head())
print(uk_10y.info())

        Date  BondYield
0 1990-01-02   9.959653
1 1990-01-03  10.011793
2 1990-01-04   9.978281
3 1990-01-05  10.013562
4 1990-01-08  10.103279
<class 'pandas.DataFrame'>
RangeIndex: 3793 entries, 0 to 3792
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       3793 non-null   datetime64[us]
 1   BondYield  3793 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 59.4 KB
None


In [110]:
df_uk = pd.merge(ftse_prices, uk_10y, on="Date", how="inner")
df_uk.head()

,Date,Price,BondYield
0,2000-04-25,6282.97,5.235517
1,2000-04-20,6241.22,5.192162
2,2000-04-19,6184.91,5.184162
3,2000-04-18,6074.04,5.176914
4,2000-04-17,5994.57,5.195414


In [112]:
check_nans(df_uk)
df_uk.describe()

Количество пропусков:
Date         0
Price        0
BondYield    0
dtype: int64

Процент пропусков:
Date         0.0
Price        0.0
BondYield    0.0
dtype: float64

Всего NaN в таблице: 0


,Date,Price,BondYield
count,2101,2101.000000,2101.000000
mean,1996-02-25 21:27:50.633031,4141.886816,7.172121
min,1992-01-02 00:00:00,2281.000000,4.122027
25%,1994-01-28 00:00:00,3039.600000,5.811979
50%,1996-02-26 00:00:00,3707.300000,7.576137
75%,1998-03-24 00:00:00,5330.800000,8.307791
max,2000-04-25 00:00:00,6930.200000,9.653103
std,NaN,1330.370897,1.396883


In [113]:
df_uk_processed = process_colums(df_uk, "UK")
df_uk_processed.describe()

1


,Date,Price,BondYield,s_UK,i_UK,delta_i_UK,Y_UK,ER_UK
count,2101,2101.000000,2101.000000,2100.000000,2101.000000,2100.000000,2100.000000,2100.000000
mean,1996-02-25 21:27:50.633031,4141.886816,7.172121,0.044021,0.019650,-0.001913,0.014344,0.024374
min,1992-01-02 00:00:00,2281.000000,4.122027,-4.139903,0.011293,-0.495470,-2.582430,-4.165363
25%,1994-01-28 00:00:00,3039.600000,5.811979,-0.504159,0.015923,-0.038281,-0.237947,-0.524240
50%,1996-02-26 00:00:00,3707.300000,7.576137,0.050252,0.020757,-0.003583,0.026876,0.031012
75%,1998-03-24 00:00:00,5330.800000,8.307791,0.583603,0.022761,0.031726,0.287104,0.561716
max,2000-04-25 00:00:00,6930.200000,9.653103,5.439552,0.026447,0.344324,3.716027,5.415035
std,NaN,1330.370897,1.396883,0.945754,0.003827,0.063909,0.479315,0.945809


In [127]:

df_uk_processed_full_time = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["full_time"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["full_time"][1])].copy()
df_uk_processed_sub1 = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["sub1"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["sub1"][1])].copy()
df_uk_processed_sub2 = df_uk_processed[(df_uk_processed["Date"] >= TIME_PERIODS["sub2"][0]) & (df_uk_processed["Date"] <= TIME_PERIODS["sub2"][1])].copy()
table_1_panel_a_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(df_uk_processed_full_time["ER_UK"]),
    "YUK": calculate_table1_stats(df_uk_processed_full_time["Y_UK"]),
    "DLFTSE": calculate_table1_stats(df_uk_processed_full_time["s_UK"]),
})
table_1_panel_b_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(df_uk_processed_sub1["ER_UK"]),
    "YUK": calculate_table1_stats(df_uk_processed_sub1["Y_UK"]),
    "DLFTSE": calculate_table1_stats(df_uk_processed_sub1["s_UK"]),
})
table_1_panel_c_uk = pd.DataFrame({
    "ERUK": calculate_table1_stats(df_uk_processed_sub2["ER_UK"]),
    "YUK": calculate_table1_stats(df_uk_processed_sub2["Y_UK"]),
    "DLFTSE": calculate_table1_stats(df_uk_processed_sub2["s_UK"]),
})
print(f"panel a:{TIME_PERIODS["full_time"]}\n")
print(table_1_panel_a_uk.round(4))
print("--------------")
print(f"panel b:{TIME_PERIODS["sub1"]}\n")
print(table_1_panel_b_uk.round(4))
print("--------------")
print(f"panel c:{TIME_PERIODS["sub2"]}\n")
print(table_1_panel_c_uk.round(4))

panel a:['1992-01-03', '2000-04-20']

               ERUK        YUK    DLFTSE
Mean         0.0241     0.0145    0.0437
Median       0.0295     0.0271    0.0484
Std. Dev.    0.9459     0.4794    0.9459
Skewness     0.0123    -0.1065    0.0068
Kurtosis     5.1148     6.8552    5.1190
JB         391.2158  1303.7991  392.7359
LB(1)       14.7495     0.3512   14.7334
LB(5)       27.8077     2.4863   27.8226
ADF(10)    -13.8993   -13.9417  -13.9041
--------------
panel b:['1992-01-03', '1997-05-30']

               ERUK       YUK    DLFTSE
Mean         0.0231    0.0116    0.0452
Median       0.0193    0.0301    0.0431
Std. Dev.    0.7441    0.5093    0.7440
Skewness     0.2449   -0.0173    0.2472
Kurtosis     6.5719    6.9952    6.5786
JB         740.3552  909.2017  743.3546
LB(1)        3.6575    0.1161    3.6372
LB(5)        4.5246    2.6352    4.4894
ADF(10)    -11.5843  -11.5826  -11.5928
--------------
panel c:['1997-06-02', '2000-04-20']

              ERUK       YUK   DLFTSE
Mean    

Теперь для Германии

In [136]:
dax = pd.read_csv("dax.csv")
dax = dax.rename(columns={"Close":"Price"})
dax = dax[["Date", "Price"]]
dax.head()

,Date,Price
0,1992-01-02,1601.9
1,1992-01-03,1603.6
2,1992-01-06,1603.3
3,1992-01-07,1592.5
4,1992-01-08,1578.7


In [137]:
dax["Date"] = pd.to_datetime(dax["Date"], format="%Y-%m-%d")
dax.describe()

,Date,Price
count,2089,2089.000000
mean,1996-02-27 15:44:22.517951,3211.733389
min,1992-01-02 00:00:00,1420.300000
25%,1994-01-28 00:00:00,2027.400000
50%,1996-02-23 00:00:00,2444.900000
75%,1998-03-30 00:00:00,4595.820000
max,2000-04-25 00:00:00,8064.970000
std,NaN,1615.600879


In [139]:
check_nans(dax)

Количество пропусков:
Date     0
Price    0
dtype: int64

Процент пропусков:
Date     0.0
Price    0.0
dtype: float64

Всего NaN в таблице: 0


In [140]:
de_10y = pd.read_csv("de10y_stooq.csv")
de_10y = de_10y.rename(columns={"date":"Date", "close":"BondYield"})
de_10y["Date"] = pd.to_datetime()


,date,close
0,1990-09-05,9.005
1,1990-09-07,9.005
2,1990-09-10,8.925
3,1990-09-11,8.954
4,1990-09-12,8.965
